# Install Dependencies and Import

In [1]:
# Install dependencies
!pip install -q --upgrade numerapi numerai-tools optuna xgboost catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 27.5 MB/s eta 0:00:00:00:0100:01


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import timedelta
import time

from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

import json
import os
import gc
import shutil
import itertools
from tqdm import tqdm
import random

import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer
import cloudpickle
import pickle
import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:.3f}'.format
pd.options.display.max_columns = 500

# Inline plots
%matplotlib inline

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Functions

In [3]:
def per_era_metrics(val_df, pred_col, calc_mmc=True, only_metrics=True):
    """Расчёт метрик по эрам: CORR и MMC"""

    # Вспомогательные функции
    def sharpe(series): 
        std = series.std(ddof=0)
        return series.mean() / std if std != 0 else np.nan

    def max_drawdown(series):
        cum = series.cumsum()
        return (cum.expanding().max() - cum).max()

    # Словарь для хранения метрик
    metrics = {}

    # CORR
    per_era_corr = val_df.groupby("era").apply(lambda x: numerai_corr(x[[pred_col]], x["target"]))

    metrics.update({
        "corr_mean": round(per_era_corr[pred_col].mean(), 3),
        "corr_std": round(per_era_corr[pred_col].std(ddof=0), 3),
        "corr_sharpe": round(sharpe(per_era_corr[pred_col]), 3),
        "corr_ ": round(max_drawdown(per_era_corr[pred_col]), 3),
    })

    # MMC
    if calc_mmc:
        per_era_mmc = val_df.dropna().groupby("era").apply(lambda x: correlation_contribution(x[[pred_col]], x["meta_model"], x["target"]))

        metrics.update({
            "mmc_mean": round(per_era_mmc[pred_col].mean(), 3),
            "mmc_std": round(per_era_mmc[pred_col].std(ddof=0), 3),
            "mmc_sharpe": round(sharpe(per_era_mmc[pred_col]), 3),
            "mmc_max_drawdown": round(max_drawdown(per_era_mmc[pred_col]), 3),
        })

    if only_metrics:
        return metrics
    if calc_mmc:
        return metrics, per_era_corr, per_era_mmc
    else:
        return metrics, per_era_corr

In [4]:
def train_and_evaluate(train_df, val_df, feature_set, model_name, params) -> tuple:

    start_time = time.time()

    # Обучение модели
    model = lgb.LGBMRegressor(**params, random_state=SEED)
    model.fit(train_df[feature_set], train_df["target"])
    print("Обучение завершено")

    # Прогноз на Validation
    val_df[model_name] = model.predict(val_df[feature_set])
    print("Валидация завершена")

    # Расчёт времени
    elapsed_time = str(timedelta(seconds=int(time.time() - start_time)))

    return elapsed_time, model

In [5]:
def era_wise_cv(train_df, feature_set, params, embargo=4, n_splits=5):
    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=embargo)

    results = []
    all_importances = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(eras), 1):
        train_eras = [eras[i] for i in train_idx]
        val_eras = [eras[i] for i in val_idx]

        print(f"\nFold {fold}: Train {train_eras[0]}–{train_eras[-1]}, Val {val_eras[0]}–{val_eras[-1]}")

        train_split = train_df[train_df["era"].isin(train_eras)].copy()
        val_split = train_df[train_df["era"].isin(val_eras)].copy()
        
        X_train, y_train = train_split[feature_set], train_split["target"]
        X_val, y_val = val_split[feature_set], val_split["target"] 


        model = lgb.LGBMRegressor(**params, random_state=SEED)
        model.fit(X_train, y_train)
        val_split["prediction"] = model.predict(X_val)

        per_era_corr = val_split.groupby("era").apply(lambda x: numerai_corr(x[["prediction"]], x["target"]))["prediction"] 
        corr_mean = per_era_corr.mean()
        corr_std = per_era_corr.std(ddof=0)
        corr_sharpe = corr_mean / corr_std if corr_std != 0 else np.nan

        results.append({
            "fold": str(fold),
            "train_eras": f"{train_eras[0]}-{train_eras[-1]}",
            "val_eras": f"{val_eras[0]}-{val_eras[-1]}",
            "corr_sharpe": round(corr_sharpe, 3),
            "corr_mean": round(corr_mean, 3),
            "corr_std": round(corr_std, 3),
        })
        

        importance = pd.DataFrame({
            "feature": feature_set,
            "importance": model.feature_importances_})
        all_importances.append(importance.set_index("feature"))


        del model
        gc.collect()


    # Сборка результатов
    cv_results = pd.DataFrame(results)
    cv_mean = cv_results.mean(numeric_only=True).round(3).to_dict()
    
    # Сборка важности признаков
    importance_df = pd.concat(all_importances, axis=1).mean(axis=1).sort_values(ascending=False).reset_index()
    importance_df.columns = ["feature", "importance"]


    return cv_results, cv_mean, importance_df

In [6]:
def era_wise_cv_optuna(train_df, feature_set, params, embargo=4, n_splits=5, esr=None):

    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=embargo)

    results = []
    best_iters = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(eras), 1):
        train_eras = [eras[i] for i in train_idx]
        val_eras = [eras[i] for i in val_idx]

        train_split = train_df[train_df["era"].isin(train_eras)]
        val_split   = train_df[train_df["era"].isin(val_eras)]

        X_train, y_train = train_split[feature_set], train_split["target"]
        X_val, y_val = val_split[feature_set], val_split["target"]

        #  TRAIN LGBM WITH EARLY STOPPING
        model = lgb.LGBMRegressor(**params, random_state=SEED)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_names=["val"],
            callbacks=[lgb.early_stopping(stopping_rounds=esr)]  # lgb.log_evaluation(period=0)
        )

        best_iters.append(model.best_iteration_)

        #  PREDICTIONS
        val_split["prediction"] = model.predict(X_val)


        per_era_corr = val_split.groupby("era").apply(lambda x: numerai_corr(x[["prediction"]], x["target"]))["prediction"]
        corr_mean = per_era_corr.mean()
        corr_std = per_era_corr.std(ddof=0)
        corr_sharpe = corr_mean / corr_std if corr_std > 0 else np.nan

        results.append(corr_sharpe)


    return {"corr_sharpe": np.mean(results)}

In [7]:
def get_feature_importance(model, normalize=False):
    
    feat_names = model.booster_.feature_name()
    gain  = model.booster_.feature_importance(importance_type="gain")
    split = model.booster_.feature_importance(importance_type="split")

    df = pd.DataFrame({"feature": feat_names, "gain": gain, "split": split})

    if normalize:
        df["gain"]  = df["gain"]  / df["gain"].sum()
        df["split"] = df["split"] / df["split"].sum()

    return df.sort_values("gain", ascending=False).reset_index(drop=True)

In [8]:
def get_diff_cv(results_df, model_name_diff, model_name_cv, cv_mean):
    row = results_df.loc[results_df["model_name"] == model_name_diff, ["model_name", "corr_sharpe", "corr_mean", "corr_std"]]
    row_cv = pd.DataFrame([{"model_name": model_name_cv, "corr_sharpe": cv_mean["corr_sharpe"], "corr_mean": cv_mean["corr_mean"], "corr_std": cv_mean["corr_std"],}])
    df_diff_cv = pd.concat([row, row_cv], ignore_index=True)

    return df_diff_cv

In [9]:
def log_to_results(results_df, model_name, elapsed_time, metrics, params):
    new_row = {"model_name": model_name, "time": elapsed_time, **metrics, **params}
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

    return results_df

In [10]:
def get_last_fold(train_df, feature_set, fold=-1):
    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=5, gap=4)
    splits = list(tscv.split(eras))
    train_idx, val_idx = splits[fold]

    last_train_eras = [eras[i] for i in train_idx]
    last_val_eras = [eras[i] for i in val_idx]

    train_es = train_df[train_df["era"].isin(last_train_eras)]
    val_es = train_df[train_df["era"].isin(last_val_eras)]

    X_train_es = train_es[feature_set]
    y_train_es = train_es["target"]
    X_val_es = val_es[feature_set]
    y_val_es = val_es["target"]

    return X_train_es, y_train_es, X_val_es, y_val_es

# Loading Numerai Datasets  

В предыдущем блокноте:
- Почистил `train` от эр с пропусками.
- Применил `validation` эмбарго 4 эры и добавил `numerai_meta_model`.
- Сгруппировал признаки по наборам для `medium`.  
  
Загружаю с Google Disk.

In [11]:
# Загрузка подготовленных датасетов
train = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/train_medium.parquet")
validation = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/validation_medium.parquet")

# Загрузка метаданных
with open('/content/drive/MyDrive/Colab Notebooks/dict_medium_clean.pkl', 'rb') as f:
    dict_medium = pickle.load(f)

# Список признаков
feature_set = list(dict_medium['all'])


print(f"Train: {train.shape}, Eras: {train['era'].min()}-{train['era'].max()}")
print(f"Validation: {validation.shape}, Eras: {validation['era'].min()}-{validation['era'].max()}")
print(f"Features: {len(feature_set)}")

Train: (1865208, 739), Eras: 210-574
Validation: (3777642, 740), Eras: 579-1187
Features: 737


In [12]:
# Датафреймы для результатов

cols = ["model_name", "time"]
cols_corr = ["corr_sharpe", "corr_mean", "corr_std", "corr_max_drawdown"]
cols_mmc = ["mmc_mean", "mmc_std", "mmc_sharpe", "mmc_max_drawdown"]


results_df = pd.DataFrame(columns=(cols + cols_corr + cols_mmc))
neut_df = results_df.copy()  # Датафрейм для нейтрализованных результатов
results_df

,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown


# Baseline Models with Large Params Numerai  
  
  

In [13]:
# LGBM Regressor 
model_name = "baseline"

params = {
    "n_estimators": 20000,
    "learning_rate": 0.001,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "verbosity": -1,
    "device_type": "gpu",
}


elapsed_time, _ = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params)
results_df

Обучение завершено
Валидация завершена


,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type
0,baseline,1:21:02,1.469,0.032,0.021,0.070,-0.002,0.010,-0.151,0.213,20000.000,0.001,6.000,64.000,0.100,-1.000,gpu


`corr_mean` = 0.032 — подозрительно высокий. Топы держат ~0.022  
`corr_sharpe` = 1.469 — хороший, но не аномальный.  
`corr_std` = 0.021 — нормальный уровень волатильности.  
`mmc_mean` = -0.002 — отрицательный! Это плохо.  

# Era-Wise CV for Baseline

In [14]:
model_name = "baseline_cv"

cv_results, cv_mean, importance_df = era_wise_cv(
    train_df=train,
    feature_set=feature_set,
    params=params,
    n_splits=5,
    embargo=4)


print("РЕЗУЛЬТАТЫ ПО ФОЛДАМ:")
display(cv_results)
print("СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:")
display(get_diff_cv(results_df, model_name_diff="baseline", model_name_cv="baseline_cv", cv_mean=cv_mean))


Fold 1: Train 210–270, Val 275–334

Fold 2: Train 210–330, Val 335–394

Fold 3: Train 210–390, Val 395–454

Fold 4: Train 210–450, Val 455–514

Fold 5: Train 210–510, Val 515–574
РЕЗУЛЬТАТЫ ПО ФОЛДАМ:


,fold,train_eras,val_eras,corr_sharpe,corr_mean,corr_std
0,1,210-270,275-334,1.528,0.031,0.020
1,2,210-330,335-394,1.918,0.029,0.015
2,3,210-390,395-454,2.511,0.041,0.016
3,4,210-450,455-514,1.802,0.034,0.019
4,5,210-510,515-574,2.772,0.045,0.016


СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:


,model_name,corr_sharpe,corr_mean,corr_std
0,baseline,1.469,0.032,0.021
1,baseline_cv,2.106,0.036,0.017


# Early Stopping on Last Fold

In [ ]:
# Получаем best_iter для последних двух фолдов
# GigaCode
esr = 100
best_iter_list = []

for fold in [-1, -2]:
    X_train_es, y_train_es, X_val_es, y_val_es = get_last_fold(train, feature_set, fold=fold)

    model = lgb.LGBMRegressor(**params, random_state=SEED)
    model.fit(
        X_train_es, y_train_es,
        eval_set=[(X_val_es, y_val_es)],
        eval_names=["val_es"],
        eval_metric="l2",
        callbacks=[early_stopping(stopping_rounds=esr)]#, log_evaluation(period=0)],  
    )

    best_iter_list.append((fold, model.best_iteration_))


best_iter = best_iter_list[0][1]

print(f"Best Iter Last Fold: {best_iter}")
print(f"Best Iter List: {best_iter_list}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[10089]	val_es's l2: 0.0496883
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[7324]	val_es's l2: 0.049729
Best Iter Last Fold: 10089
Best Iter List: [(-1, 10089), (-2, 7324)]


# Tuned Params

In [ ]:
# Holdout Validation with Best Iteration + Регуляризация
# GigaCode
model_name = "tuned"

params_tuned = {
    'n_estimators': int(best_iter * 1.2),       # как просили — берём best_iter
    'learning_rate': 0.01,           # ↑ в 10 раз → быстрее сходимость
    'max_depth': 5,                  # ↓ сложность дерева
    'num_leaves': 31,                # соответствует max_depth=5 (2^5=32 → 31)
    'colsample_bytree': 0.05,        # ↓ ещё сильнее → меньше leakage
    'subsample': 0.8,                # сэмплирование строк → регуляризация
    'reg_alpha': 10.0,               # L1 регуляризация → обнуляет слабые фичи
    'reg_lambda': 10.0,              # L2 → сглаживает веса
    'min_child_samples': 200,        # ↑ сильно → не даёт переобучаться на шум
    'min_child_weight': 1e-2,        # ↑ порог разбиения
    'verbosity': -1,
    'device_type': 'gpu',
}

elapsed_time, _ = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_tuned)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_tuned)
results_df

Обучение завершено
Валидация завершена


,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type,subsample,reg_alpha,reg_lambda,min_child_samples,min_child_weight
0,baseline,1:21:02,1.469,0.032,0.021,0.070,-0.002,0.010,-0.151,0.213,20000.000,0.001,6.000,64.000,0.100,-1.000,gpu,NaN,NaN,NaN,NaN,NaN
1,tuned,0:20:34,1.532,0.031,0.020,0.077,-0.001,0.009,-0.102,0.141,10089.000,0.010,5.000,31.000,0.050,-1.000,gpu,0.800,10.000,10.000,200.000,0.010


In [23]:
# Era-Wise CV for Tuned Params
model_name = "tuned_cv"

cv_results, cv_mean, importance_df = era_wise_cv(
    train_df=train,
    feature_set=feature_set,
    params=params_tuned,
    n_splits=5,
    embargo=4)


print("РЕЗУЛЬТАТЫ ПО ФОЛДАМ:")
display(cv_results)
print("СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:")
display(get_diff_cv(results_df, model_name_diff="tuned", model_name_cv="tuned_cv", cv_mean=cv_mean))


Fold 1: Train 210–270, Val 275–334

Fold 2: Train 210–330, Val 335–394

Fold 3: Train 210–390, Val 395–454

Fold 4: Train 210–450, Val 455–514

Fold 5: Train 210–510, Val 515–574
РЕЗУЛЬТАТЫ ПО ФОЛДАМ:


,fold,train_eras,val_eras,corr_sharpe,corr_mean,corr_std
0,1,210-270,275-334,1.688,0.030,0.018
1,2,210-330,335-394,1.975,0.030,0.015
2,3,210-390,395-454,2.506,0.040,0.016
3,4,210-450,455-514,1.960,0.034,0.017
4,5,210-510,515-574,2.701,0.040,0.015


СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:


,model_name,corr_sharpe,corr_mean,corr_std
0,tuned,1.532,0.031,0.020
1,tuned_cv,2.166,0.035,0.016


# ChatGPT

In [24]:
# Получаем best_iter для последних двух фолдов
esr = 500

params_gpt = {
    "n_estimators": 20000,
    "learning_rate": 0.001,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "verbosity": -1,
    "device_type": "gpu",
}

X_train_es, y_train_es, X_val_es, y_val_es = get_last_fold(train, feature_set, fold=-1)

model = lgb.LGBMRegressor(**params_gpt, random_state=SEED)
model.fit(
    X_train_es, y_train_es,
    eval_set=[(X_val_es, y_val_es)],
    eval_names=["val_es"],
    eval_metric="l2",
    callbacks=[early_stopping(stopping_rounds=esr)]
)


best_iter_gpt = best_iter_list[0][1]

print(f"Best Iter Last Fold: {best_iter}")
print(f"Best Iter List: {best_iter_list}")

Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[11514]	val_es's l2: 0.0496864
Best Iter Last Fold: 10089
Best Iter List: [(-1, 10089), (-2, 7324)]


In [27]:
best_iter_gpt = model.best_iteration_


In [29]:
# Holdout Validation with Best Iteration + Регуляризация
# ChatGPT
model_name = "tuned_gpt"

params_gpt = {
    "n_estimators": best_iter_gpt,       # как просили — берём best_iter
    "learning_rate": 0.001,
    "max_depth": 7,
    "num_leaves": 128,
    "colsample_bytree": 0.2,
    "min_child_samples": 50,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "verbosity": -1,
    "device_type": "gpu"
}

elapsed_time, _ = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_gpt)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_tuned)
results_df

Обучение завершено
Валидация завершена


,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type,subsample,reg_alpha,reg_lambda,min_child_samples,min_child_weight
0,baseline,1:21:02,1.469,0.032,0.021,0.070,-0.002,0.010,-0.151,0.213,20000.000,0.001,6.000,64.000,0.100,-1.000,gpu,NaN,NaN,NaN,NaN,NaN
1,tuned,0:20:34,1.532,0.031,0.020,0.077,-0.001,0.009,-0.102,0.141,10089.000,0.010,5.000,31.000,0.050,-1.000,gpu,0.800,10.000,10.000,200.000,0.010
2,tuned_gpt,0:55:51,1.483,0.032,0.021,0.064,-0.001,0.011,-0.096,0.190,10089.000,0.010,5.000,31.000,0.050,-1.000,gpu,0.800,10.000,10.000,200.000,0.010


In [30]:
with open("/content/drive/MyDrive/Colab Notebooks/val_med_pred.pkl", "wb") as f:
    pickle.dump(validation, f)
# print(f"Validation с предсказаниями сохранён в {validation_file}")

with open("/content/drive/MyDrive/Colab Notebooks/results_med.pkl", "wb") as f:
    pickle.dump(results_df, f)

def era_wise_cv_fold(train_df, features, params, target_col='target', pred_col='prediction',
                     n_splits=5, purge=2, embargo=2, early_stopping_rounds=100):
    """
    Выполняет обучение модели по фолдам и возвращает OOF-предсказания и метрики по фолдам.
    """
    X, y = train_df[features], train_df[target_col]
    eras = train_df['era'].values

    cv = PurgedGroupTimeSeriesSplit(n_splits=n_splits, purge=purge, embargo=embargo)

    oof = train_df[["era", "target"]].copy()
    oof[pred_col] = 0.0

    fold_results = []
    all_importances = []
    best_iters = {}

    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, eras)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        train_eras = eras[train_idx]
        val_eras = eras[val_idx]
        train_era_start, train_era_end = train_eras[0], train_eras[-1]
        val_era_start, val_era_end = val_eras[0], val_eras[-1]

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            eval_names=['train', 'val'],
            eval_metric='l2',
            callbacks=[
                lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False),
                lgb.log_evaluation(period=0)
            ],
        )

        best_iter = model.best_iteration_ if hasattr(model, 'best_iteration_') else None
        best_iters[fold + 1] = best_iter

        val_pred = model.predict(X_val)
        oof.loc[oof.index[val_idx], pred_col] = val_pred
        oof_val = oof.iloc[val_idx].copy()

        metrics_corr = compute_metrics_corr(oof_val, pred_col=pred_col, target_col=target_col)
        feature_exposure = compute_feature_exposure(predictions=pd.Series(val_pred, index=X_val.index), features=X_val)

        fold_results.append({
            "fold": fold + 1,
            "corr_mean": metrics_corr["corr_mean"],
            "corr_std": metrics_corr["corr_std"],
            "corr_sharpe": metrics_corr["corr_sharpe"],
            "max_drawdown": metrics_corr["corr_max_drawdown"],
            "feature_exposure": feature_exposure,
            "train_eras": f"{train_era_start}-{train_era_end}",
            "val_eras": f"{val_era_start}-{val_era_end}",
            "val_size": len(val_idx),
            "best_iter": best_iter,
        })

        importance = pd.DataFrame({
            "feature": features,
            "gain": model.booster_.feature_importance(importance_type='gain')
        })
        all_importances.append(importance.set_index("feature"))

    return oof, fold_results, all_importances, best_iters


Продолжаем участие в классическом турнире Numerai.  
Уже сделано:
1. Устновлены зависимости и импорты. В VS Code используется ядро jupyter-сервера, запущенно в ячейке Google Colab with GPU (платная подписка).
2. Загружены подготовленные train и validation.
3. Провел кросс-валидацию на Train и получил метрики.

Теперь давай создадим Optuna:
- Для CV используй era_wise_cv_fold
- Пространство: небольшие изменения вокруг baseline
- study.enqueue_trial(standard_large_lgbm_params)
- Objective: maximize corr_mean
- Валидация: early_stopping, фиксированные фолды
- Логирование: baseline_score в user_attr
- Получить top-3 trial по `corr_mean`




Продолжаем участие в классическом турнире Numerai. 
**Цель этого блокнота** - выбрать модель с лучшими параметрами. Ансамблем, нейтрализацией и всем остальным мы займемся в следующем блокноте.

**План блокнота**
0. Установка зависимостей и импорты
1. Загрузка данных
   - В предыдущем блокноте я уже почистил от пропусков и подготовил Датасеты Numerai для обучения
   - 737 features - набор Medium
   - Train: 1.8M строк, Столбцы: id в индексе, era, target, 737 features
   - Validation: 3.8M строк, Столбцы: id в индексе, era, target, numerai_meta_model (частично), 737 features
   - dict_medium - словарь набора признаков, сгруппированный по Medium

2. Baseline CV (PurgedGroupTimeSeriesSplit)
   - Параметры: standard_large_lgbm_params
   - early_stopping_rounds
   - Сбор OOF по фолдам
   - feature importance по фолдам для расчёта FNC
   - Метрики:
      - corr_mean
      - corr_std
      - corr_sharpe
      - corr_max_drawdown
      - autocor
      - feature_exposure
      - fnc (нейтрализация по топ-5 признакам по gain)
   - Графики: 
      - Plot the per-era correlation (линия + диапазон)
      - Plot the cumulative per-era correlation
      - Distribution of Era Correlations (гистограмма)
      - Feature Importance (Top 20 by Gain)
      - Prediction Distribution (валидация vs трейн)
      - Autocorrelation Plot (lag-1)
   - Сохранение: oof_baseline.parquet, metrics_baseline.json

3. Optuna + CV (те же фолды)
   - study.enqueue_trial(standard_large_lgbm_params)
   - Пространство: вокруг baseline
   - Objective: maximize corr_mean
   - Валидация: early_stopping, фиксированные фолды
   - Логирование: baseline_score в user_attr
   - Получить top-3 trial по `corr_mean`

4. CV top-3 trial Optuna best_params
   - Для каждого: полный CV → сбор OOF, importance , best_iteration (усечённое среднее без 1-2 первых фолдов)
   - Полная оценка по всем метрикам и ыбор одной лучшей модели по балансу
      - Высокий corr_mean
      - corr_sharpe > 1.0
      - autocorr < 0.25
      - feature_exposure < 0.2
      - низкий max_drawdown
   - Сохранение: best_iteration.json, feature_importance_cv.parquet

5. Финальная модель
   - Обучение на всём Train
   - Параметры: best_params, n_estimators = truncated_mean_best_iter
   - Предсказание на Validation
   - Расчёт всех метрик на Validation + набор MMC: mmc_mean, mmc_std, mmc_sharpe, mmc_max_drawdown
   - Графики: 
      - Plot the per-era correlation (линия + диапазон)
      - Plot the cumulative per-era correlation
      - Plot the per-era correlation MMC
      - Plot the cumulative per-era MMC
      - Distribution of Era Correlations (гистограмма)
      - Feature Importance (Top 20 by Gain)
      - Prediction Distribution (валидация vs трейн)
      - Autocorrelation Plot (lag-1)
   - feature importance
   - Сохранение: metrics_final.json, feature_importance_final.parquet

6. Сохранение результатов
   - Все прогнозы: oof_baseline.parquet, oof_optuna.parquet, val_predictions.parquet
   - Параметры: best_params.json
   - Метрики: metrics_baseline.json, metrics_final.json
   - Важность: feature_importance_cv.parquet, feature_importance_final.parquet


Теперь мы с тобой будем пошагово реализовывать на план. Прими к сведению следующие моменты:
- Это один из блокнотов, в которых я создаю рабочую модель для отправки на турнир Numerai. А также буду загружать его на GitHub для портфолио своих работ, чтобы устроиться на работу.
- Офрмить ноутбук нужно по всем правилам и лучшим практикам, как с точки зрения раработки ML-решений, так и в целом написания кода на puthon
- Учитывай специфику классического турнира Numerai на декабрь 2025 года.
- Т.к. это исследовательский блокнот, то нужно его правильно формить с заголовками пояснениями и выводами в markdawn ячейках. А также подробное комментирование кода.

Мы уже выполнили шаги:
0. Установка зависимостей и импорты
1. Загрузка данных: train, validation, dict_medium. Пока сделали облешченную версию датасетов для DEBUG-режима (каждая 20 эра и только 20 признаков)


Сейчас твоя задача написать код для расчета всех метрик. На разных этапах мы будем считать разные метрики, поэтому это должно быть универсально и переносимо

Ранее мы уже с тобой создали класс PurgedTimeSeriesSplit.

```
class PurgedTimeSeriesSplit:
    """
    Временное разбиение с purge и embargo.
    Гарантирует ровно n_folds фолдов.
    """
    def __init__(self, n_folds=5, purge_gap=1, embargo=1):
        self.n_folds = n_folds
        self.purge_gap = purge_gap
        self.embargo = embargo

    def split(self, X, y=None, groups=None):
        # Уникальные эры в хронологическом порядке
        unique_eras = np.unique(groups)
        era_to_order = {era: i for i, era in enumerate(unique_eras)}
        group_order = np.array([era_to_order[era] for era in groups])

        n_eras = len(unique_eras)

        # Делим на (n_folds + 1) блоков: чтобы было n_folds валидаций
        split_indices = np.array_split(np.arange(n_eras), self.n_folds + 1)

        for i in range(1, self.n_folds + 1):
            if i >= len(split_indices):
                break

            # Обучающие эры: все блоки до i
            train_era_indices = np.concatenate(split_indices[:i])
            train_eras = unique_eras[train_era_indices]

            # Валидационные эры: i-й блок
            val_era_indices = split_indices[i]
            val_eras = unique_eras[val_era_indices]

            # Purge: удаляем последние `purge_gap` эр из train
            train_max_order = max(era_to_order[era] for era in train_eras)
            purged_max_order = train_max_order - self.purge_gap

            # Embargo: пропускаем первые `embargo` эр после purged_max_order
            min_val_order = purged_max_order + 1 + self.embargo

            # Оставляем только те val-эры, которые после embargo
            allowed_val_eras = [era for era in val_eras if era_to_order[era] >= min_val_order]

            # пропускаем, если валидация пуста
            if len(allowed_val_eras) == 0:
                continue  

            # Маски по данным
            train_mask = group_order <= purged_max_order
            val_mask = np.isin(groups, allowed_val_eras)

            train_indices = np.where(train_mask)[0]
            val_indices = np.where(val_mask)[0]

            yield train_indices, val_indices
```

Теперь давай напишем функцию кросс-валидации с использованием PurgedTimeSeriesSplit. Эта функция должна быть универсальной чтобы использовать ее и для baseline и для Optuna

In [ ]:
# --- 2. METRICS EVALUATION MODULE (Dec 2025) ---
"""
Модуль для расчёта всех метрик в соответствии с текущими стандартами Numerai (Dec 2025).
Использует numerai_corr как основу.
"""

import numpy as np
import pandas as pd
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize


def compute_per_era_corr(
    predictions: pd.Series,
    targets: pd.Series,
    era_col: pd.Series,
    pred_col: str = "prediction"
) -> pd.Series:
    """
    Вычисляет per-era корреляцию с использованием numerai_corr (стандарт Numerai, Dec 2025).
    
    Args:
        predictions: предсказания (pd.Series)
        targets: таргеты (pd.Series)
        era_col: эры (pd.Series)
        pred_col: имя колонки предсказания (по умолчанию 'prediction')
    
    Returns:
        pd.Series: корреляции по каждой эре
    """
    df = pd.DataFrame({
        pred_col: predictions.values,
        "target": targets.values,
        "era": era_col.values
    })
    per_era_corr = df.groupby("era").apply(
        lambda x: numerai_corr(x[[pred_col]], x["target"])
    )
    return per_era_corr


def compute_corr_metrics(correlations: pd.Series) -> dict:
    """
    Вычисляет базовые метрики по вектору корреляций.
    
    Args:
        correlations: pd.Series корреляций по эрам
    
    Returns:
        dict: corr_mean, corr_std, corr_sharpe, corr_max_drawdown
    """
    corr_mean = float(correlations.mean())
    corr_std = float(correlations.std())
    corr_sharpe = corr_mean / corr_std if corr_std != 0 else 0.0

    # Max Drawdown (на основе кумулятивной корреляции)
    cumcorr = correlations.cumsum()
    running_max = cumcorr.cummax()
    drawdowns = running_max - cumcorr
    corr_max_drawdown = float(drawdowns.max())

    return {
        "corr_mean": corr_mean,
        "corr_std": corr_std,
        "corr_sharpe": corr_sharpe,
        "corr_max_drawdown": corr_max_drawdown
    }


def compute_autocorr(
    predictions: pd.Series,
    era_col: pd.Series
) -> float:
    """
    Средняя автокорреляция (lag=1) средних предсказаний по эрам.
    
    Args:
        predictions: предсказания
        era_col: столбец эр
    
    Returns:
        float: autocorrelation (lag-1)
    """
    df = pd.DataFrame({'pred': predictions, 'era': era_col.astype(str)})
    df = df.groupby('era')['pred'].mean().reset_index()
    df = df.sort_values('era')
    pred_mean = df['pred'].values
    
    if len(pred_mean) < 2:
        return 0.0
    
    autocov = np.correlate(pred_mean[:-1] - pred_mean.mean(), pred_mean[1:] - pred_mean.mean(), mode='valid')
    var = np.var(pred_mean)
    return float(autocov[0] / var) if var != 0 else 0.0


def compute_feature_exposure(
    predictions: pd.Series,
    features: pd.DataFrame
) -> float:
    """
    Средняя абсолютная корреляция предсказаний с признаками.
    
    Args:
        predictions: предсказания
        features: признаки (DataFrame)
    
    Returns:
        float: feature_exposure
    """
    exposures = []
    pred = predictions.rank(pct=True, method="first")  # как в numerai
    for col in features.columns:
        feat = features[col].rank(pct=True, method="first")
        corr = np.corrcoef(pred, feat)[0, 1]
        exposures.append(abs(corr))
    return float(np.mean(exposures))


def compute_fnc(
    predictions: pd.Series,
    targets: pd.Series,
    features: pd.DataFrame,
    era_col: pd.Series,
    top_k: int = 5,
    method: str = "exposure"
) -> float:
    """
    Feature Neutral Correlation.
    Нейтрализует предсказания по top_k признакам, затем считает numerai_corr с target.
    
    Args:
        predictions: предсказания
        targets: таргеты
        features: признаки
        era_col: эры
        top_k: число признаков для нейтрализации
        method: 'gain' или 'exposure' — как выбрать топ-признаки
    
    Returns:
        float: средняя FNC по эрам
    """
    if method == "exposure":
        # Выбираем признаки с наибольшей корреляцией с pred
        pred_rank = predictions.rank(pct=True, method="first")
        exposures = features.corrwith(pred_rank, method=lambda x, y: np.corrcoef(x, y)[0, 1])
        top_features = exposures.abs().sort_values(ascending=False).head(top_k).index.tolist()
    else:
        # method == 'gain' — ожидается, что importance будет передано отдельно
        # Здесь заглушка: берём первые k
        top_features = features.columns[:top_k].tolist()

    # Нейтрализация
    try:
        df_pred = pd.DataFrame({"pred": predictions})
        neutralizers = features[top_features]
        pred_neutral = neutralize(
            df=df_pred,
            columns=["pred"],
            neutralizers=neutralizers,
            proportion=1.0,
            normalize=True,
            method="normal"
        )["pred"]
    except Exception as e:
        print(f"FNC neutralization failed: {e}")
        pred_neutral = predictions

    # Пересчитываем per-era FNC с numerai_corr
    fnc_scores = compute_per_era_corr(
        predictions=pred_neutral,
        targets=targets,
        era_col=era_col,
        pred_col="pred"
    )
    return float(fnc_scores.mean())


def compute_mmc(
    predictions: pd.Series,
    targets: pd.Series,
    meta_model: pd.Series,
    era_col: pd.Series
) -> dict:
    """
    Meta Model Contribution (MMC) — numerai-style.
    
    Args:
        predictions: предсказания
        targets: таргеты
        meta_model: numerai_meta_model (частично заполнен)
        era_col: эры
    
    Returns:
        dict: mmc_mean, mmc_std, mmc_sharpe, mmc_max_drawdown
    """
    mmc_scores = []
    df = pd.DataFrame({
        "pred": predictions,
        "target": targets,
        "meta": meta_model,
        "era": era_col
    })

    for era in df["era"].unique():
        era_data = df[df["era"] == era]
        if len(era_data) < 2 or era_data["meta"].isna().all():
            continue
        try:
            mmc = correlation_contribution(
                era_data["pred"].values,
                era_data["meta"].values,
                era_data["target"].values
            )
            mmc_scores.append(mmc)
        except:
            mmc_scores.append(0.0)

    if not mmc_scores:
        mmc_scores = [0.0]

    mmc_mean = float(np.mean(mmc_scores))
    mmc_std = float(np.std(mmc_scores))
    mmc_sharpe = mmc_mean / mmc_std if mmc_std != 0 else 0.0

    cum_mmc = np.cumsum(mmc_scores)
    running_max = np.maximum.accumulate(cum_mmc)
    drawdowns = running_max - cum_mmc
    mmc_max_drawdown = float(drawdowns.max()) if len(drawdowns) > 0 else 0.0

    return {
        "mmc_mean": mmc_mean,
        "mmc_std": mmc_std,
        "mmc_sharpe": mmc_sharpe,
        "mmc_max_drawdown": mmc_max_drawdown
    }


def evaluate_predictions(
    predictions: pd.Series,
    targets: pd.Series,
    features: pd.DataFrame,
    era_col: pd.Series,
    meta_model: pd.Series = None,
    feature_importance: dict = None,
    fnc_top_k: int = 5,
    fnc_method: str = "exposure",  # теперь по умолчанию — по корреляции с pred
    pred_col: str = "prediction"
) -> dict:
    """
    Единая точка оценки модели — все метрики за один вызов.
    
    Args:
        predictions: предсказания
        targets: таргеты
        features: признаки
        era_col: эры
        meta_model: numerai_meta_model (опционально)
        feature_importance: dict (name → importance), если fnc_method='gain'
        fnc_top_k: число признаков для FNC
        fnc_method: 'exposure' (по corr с pred) или 'gain' (по importance)
        pred_col: имя колонки предсказания (для numerai_corr)
    
    Returns:
        dict: все метрики
    """
    # 1. Per-era corr
    per_era_corr = compute_per_era_corr(predictions, targets, era_col, pred_col)
    corr_metrics = compute_corr_metrics(per_era_corr)

    # 2. Autocorr
    autocorr = compute_autocorr(predictions, era_col)

    # 3. Feature Exposure
    feat_exp = compute_feature_exposure(predictions, features)

    # 4. FNC
    fnc = compute_fnc(
        predictions, targets, features, era_col,
        top_k=fnc_top_k, method=fnc_method
    )

    # 5. MMC
    mmc_metrics = compute_mmc(predictions, targets, meta_model, era_col) if meta_model is not None else {
        "mmc_mean": 0.0,
        "mmc_std": 0.0,
        "mmc_sharpe": 0.0,
        "mmc_max_drawdown": 0.0
    }

    # Итог
    results = {
        **corr_metrics,
        "autocorr": autocorr,
        "feature_exposure": feat_exp,
        "fnc": fnc,
        "n_eras": int(era_col.nunique()),
        "n_samples": int(len(predictions))
    }
    results.update({f"mmc_{k}": v for k, v in mmc_metrics.items()})

    return results
